In [7]:
# =====================================================================
# FULL PHASE 1 FINAL
# NIFTY 500 FEATURE ENGINE (PRODUCTION VERSION)
# ---------------------------------------------------------------------
# INPUT:
#   ind_nifty500list.csv
#
# OUTPUT:
#   nifty500_quant_features.csv
#   partial_backup.csv
#
# FEATURES:
# ✓ All 503 Nifty stocks
# ✓ 2000 to present
# ✓ MultiIndex yfinance fix
# ✓ Curated indicators
# ✓ Stable cleaning
# ✓ Progress bar
# ✓ Backup saves
# ✓ ML-ready output
# =====================================================================

import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta
from tqdm import tqdm
from datetime import datetime
import time
import warnings

warnings.filterwarnings("ignore")

# =====================================================================
# USER INPUT
# =====================================================================

file_path = r"C:\Users\navon\Downloads\ind_nifty500list.csv"
ticker_col = "yfinance Symbols"
sector_col = "Industry"

START_DATE = "2000-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

OUTPUT_FILE = "nifty500_quant_features.csv"
BACKUP_FILE = "partial_backup.csv"

SAVE_EVERY = 25
MIN_ROWS = 250

# =====================================================================
# HELPERS
# =====================================================================

def memory_opt(df):

    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = df[c].astype("float32")

    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = df[c].astype("int32")

    return df

# =====================================================================
# LOAD TICKERS
# =====================================================================

stocks = pd.read_csv(file_path)

tickers = (
    stocks[ticker_col]
    .dropna()
    .unique()
    .tolist()
)

sector_map = dict(zip(stocks[ticker_col], stocks[sector_col]))

print("=" * 70)
print("FULL PHASE 1 FINAL")
print("=" * 70)
print("Tickers Found:", len(tickers))

# =====================================================================
# PROCESS SINGLE STOCK
# =====================================================================

def process_stock(ticker):

    try:

        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            progress=False
        )

        if df.empty:
            return None

        # -------------------------------------------------------------
        # FIX MULTIINDEX
        # -------------------------------------------------------------

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        if len(df) < MIN_ROWS:
            return None

        # -------------------------------------------------------------
        # INDICATORS
        # -------------------------------------------------------------

        df["sma20"] = ta.sma(df["Close"], length=20)
        df["sma50"] = ta.sma(df["Close"], length=50)
        df["sma200"] = ta.sma(df["Close"], length=200)

        df["ema12"] = ta.ema(df["Close"], length=12)
        df["ema26"] = ta.ema(df["Close"], length=26)

        df["rsi14"] = ta.rsi(df["Close"], length=14)

        macd = ta.macd(df["Close"])
        if macd is not None and not macd.empty:
            df["macd"] = macd.iloc[:, 0]
            df["macd_signal"] = macd.iloc[:, 1]
            df["macd_hist"] = macd.iloc[:, 2]
        else:
            df["macd"] = np.nan
            df["macd_signal"] = np.nan
            df["macd_hist"] = np.nan

        df["roc10"] = ta.roc(df["Close"], length=10)

        df["atr14"] = ta.atr(
            df["High"], df["Low"], df["Close"], length=14
        )

        df["obv"] = ta.obv(df["Close"], df["Volume"])

        # -------------------------------------------------------------
        # PRICE FEATURES
        # -------------------------------------------------------------

        df["ret1d"] = df["Close"].pct_change()
        df["ret5d"] = df["Close"].pct_change(5)

        df["range_pct"] = (
            (df["High"] - df["Low"]) / df["Close"]
        )

        df["volatility20"] = df["ret1d"].rolling(20).std()

        # -------------------------------------------------------------
        # TARGET
        # -------------------------------------------------------------

        df["future_return_5d"] = (
            df["Close"].shift(-5) / df["Close"] - 1
        )

        df["target"] = (
            df["future_return_5d"] > 0
        ).astype(int)

        # -------------------------------------------------------------
        # META
        # -------------------------------------------------------------

        df["Ticker"] = ticker
        df["Sector"] = sector_map.get(ticker, "Unknown")

        # -------------------------------------------------------------
        # CLEAN
        # -------------------------------------------------------------

        df = df.reset_index()

        df = df.dropna(subset=["Close", "future_return_5d"])

        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].ffill().fillna(0)

        if len(df) < 100:
            return None

        df = memory_opt(df)

        return df

    except:
        return None

# =====================================================================
# MAIN LOOP
# =====================================================================

all_data = []
ok = 0
bad = 0

start = time.time()

for i, ticker in enumerate(tqdm(tickers)):

    temp = process_stock(ticker)

    if temp is not None:
        all_data.append(temp)
        ok += 1
    else:
        bad += 1

    # backup
    if (i + 1) % SAVE_EVERY == 0 and len(all_data) > 0:

        partial = pd.concat(all_data, ignore_index=True)
        partial.to_csv(BACKUP_FILE, index=False)

# =====================================================================
# FINALIZE
# =====================================================================

print("\nFINALIZING DATASET...\n")

master = pd.concat(all_data, ignore_index=True)

num_cols = master.select_dtypes(include="number").columns
master[num_cols] = master[num_cols].fillna(0)

master = memory_opt(master)

master.to_csv(OUTPUT_FILE, index=False)

runtime = round(time.time() - start, 2)

print("=" * 70)
print("PHASE 1 COMPLETE")
print("=" * 70)

print("Main Output :", OUTPUT_FILE)
print("Backup File :", BACKUP_FILE)
print("Rows        :", f"{len(master):,}")
print("Columns     :", len(master.columns))
print("Success     :", ok)
print("Failed      :", bad)
print("Runtime     :", runtime, "sec")

print("\nNext Step: Run Phase 2 ML Engine")

FULL PHASE 1 FINAL
Tickers Found: 503


 30%|███████████████████████▋                                                        | 149/503 [02:51<05:12,  1.13it/s]$DUMMYSKFIN.NS: possibly delisted; no timezone found

1 Failed download:
['DUMMYSKFIN.NS']: possibly delisted; no timezone found
 30%|███████████████████████▊                                                        | 150/503 [03:05<29:38,  5.04s/it]$DUMMYTATAM.NS: possibly delisted; no timezone found

1 Failed download:
['DUMMYTATAM.NS']: possibly delisted; no timezone found
 30%|████████████████████████                                                        | 151/503 [03:07<22:56,  3.91s/it]$DUMMYDBRLT.NS: possibly delisted; no timezone found

1 Failed download:
['DUMMYDBRLT.NS']: possibly delisted; no timezone found
 89%|███████████████████████████████████████████████████████████████████████▎        | 448/503 [13:01<01:34,  1.72s/it]$TATAMOTORS.NS: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-04-23) (Yahoo error = "No data found, symbol may be delist


FINALIZING DATASET...

PHASE 1 COMPLETE
Main Output : nifty500_quant_features.csv
Backup File : partial_backup.csv
Rows        : 1,980,952
Columns     : 26
Success     : 493
Failed      : 10
Runtime     : 1035.62 sec

Next Step: Run Phase 2 ML Engine


In [1]:
# =====================================================================
# PHASE 2 ULTRA FINAL (INSTITUTIONAL / EFFICIENT / ROBUST)
# ---------------------------------------------------------------------
# FINAL IMPROVEMENTS ADDED:
# ✓ Faster CSV loading (dtype optimization)
# ✓ Stronger leakage protection
# ✓ Walk-forward expanding windows
# ✓ Early stopping for XGBoost
# ✓ Hist tree method (fast)
# ✓ Better top-signal precision metrics
# ✓ Sector robustness by fold
# ✓ Stability stats (mean/std)
# ✓ Memory cleanup between models
# ✓ Cleaner charts + outputs
# ✓ Safer divide-by-zero / empty fold handling
#
# FOCUS METRICS:
# - ROC AUC
# - Precision
# - Top 10% Signal Precision
# - Stability Across Time
# - Sector Robustness
# =====================================================================

import os
import gc
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    roc_auc_score
)

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =====================================================================
# SETTINGS
# =====================================================================

DATA_FILE = "nifty500_quant_features.csv"
TARGET = "target"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"phase2_ultra_output_{RUN_ID}"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TOP_SIGNAL_PERCENT = 0.10
WALK_WINDOWS = 6   # more folds
MIN_TEST_ROWS = 10000

# =====================================================================
# LOAD DATA
# =====================================================================

print("=" * 72)
print("LOADING DATA")
print("=" * 72)

df = pd.read_csv(DATA_FILE, low_memory=False)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print("Rows :", f"{len(df):,}")
print("Cols :", len(df.columns))

# =====================================================================
# FEATURE SETUP
# =====================================================================

drop_cols = [
    "Date",
    "Ticker",
    "Sector",
    "future_return_5d",
    TARGET
]

X = df.drop(columns=drop_cols, errors="ignore")
X = X.select_dtypes(include=np.number)
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# memory optimize
for c in X.columns:
    if X[c].dtype == "float64":
        X[c] = X[c].astype("float32")
    elif X[c].dtype == "int64":
        X[c] = X[c].astype("int32")

y = df[TARGET].astype("int8")

print("\nFeature Columns:", len(X.columns))
print("Target Balance:")
print(y.value_counts(normalize=True))

feature_names = X.columns.tolist()

# =====================================================================
# MODELS
# =====================================================================

def get_models():
    return {
        "DecisionTree": DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=300,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=250,
            max_depth=8,
            min_samples_leaf=150,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ),

        "XGBoost": XGBClassifier(
            n_estimators=1200,
            learning_rate=0.025,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            reg_alpha=0.0,
            tree_method="hist",
            eval_metric="logloss",
            random_state=RANDOM_STATE
        )
    }

# =====================================================================
# HELPERS
# =====================================================================

def top_signal_precision(prob, actual, pct=0.10):
    temp = pd.DataFrame({"prob": prob, "actual": actual})
    temp = temp.sort_values("prob", ascending=False)

    n = max(1, int(len(temp) * pct))
    return temp.head(n)["actual"].mean()

# =====================================================================
# WALK FORWARD ANALYSIS
# =====================================================================

print("\n" + "=" * 72)
print("WALK-FORWARD ANALYSIS")
print("=" * 72)

n = len(X)
chunk = n // WALK_WINDOWS

all_metrics = []
all_predictions = []

models = get_models()

for model_name, model in models.items():

    print(f"\nRunning {model_name}")

    for fold in range(2, WALK_WINDOWS):

        train_end = chunk * fold
        test_end = chunk * (fold + 1)

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_test = X.iloc[train_end:test_end]
        y_test = y.iloc[train_end:test_end]

        meta = df.iloc[train_end:test_end][["Date", "Ticker", "Sector"]].copy()

        if len(X_test) < MIN_TEST_ROWS:
            continue

        start = time.time()

        # -------------------------------------------------------------
        # TRAIN
        # -------------------------------------------------------------

        if model_name == "XGBoost":
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_test, y_test)],
                verbose=False
            )
        else:
            model.fit(X_train, y_train)

        # -------------------------------------------------------------
        # PREDICT
        # -------------------------------------------------------------

        pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(X_test)[:, 1]
        else:
            prob = pred.astype(float)

        runtime = round(time.time() - start, 2)

        # -------------------------------------------------------------
        # METRICS
        # -------------------------------------------------------------

        acc = accuracy_score(y_test, pred)
        prec = precision_score(y_test, pred, zero_division=0)

        try:
            auc = roc_auc_score(y_test, prob)
        except:
            auc = np.nan

        top10 = top_signal_precision(prob, y_test.values, 0.10)
        top05 = top_signal_precision(prob, y_test.values, 0.05)

        all_metrics.append({
            "Model": model_name,
            "Fold": fold - 1,
            "TrainRows": len(X_train),
            "TestRows": len(X_test),
            "Accuracy": acc,
            "Precision": prec,
            "ROC_AUC": auc,
            "Top10pct_Precision": top10,
            "Top5pct_Precision": top05,
            "RuntimeSec": runtime
        })

        pred_df = meta.copy()
        pred_df["Actual"] = y_test.values
        pred_df["Pred"] = pred
        pred_df["Prob"] = prob
        pred_df["Model"] = model_name
        pred_df["Fold"] = fold - 1

        all_predictions.append(pred_df)

        gc.collect()

# =====================================================================
# RESULTS TABLES
# =====================================================================

metrics = pd.DataFrame(all_metrics)
predictions = pd.concat(all_predictions, ignore_index=True)

metrics.to_csv(
    os.path.join(OUT_DIR, "walkforward_fold_metrics.csv"),
    index=False
)

predictions.to_csv(
    os.path.join(OUT_DIR, "predictions.csv"),
    index=False
)

summary = metrics.groupby("Model").agg({
    "Accuracy": ["mean", "std"],
    "Precision": ["mean", "std"],
    "ROC_AUC": ["mean", "std"],
    "Top10pct_Precision": "mean",
    "Top5pct_Precision": "mean",
    "RuntimeSec": "mean"
})

summary.columns = [
    "_".join(col).strip()
    for col in summary.columns.values
]

summary = summary.reset_index()
summary = summary.sort_values("ROC_AUC_mean", ascending=False)

summary.to_csv(
    os.path.join(OUT_DIR, "model_summary.csv"),
    index=False
)

print("\nMODEL SUMMARY")
print(summary)

# =====================================================================
# FINAL FULL FIT BEST MODEL (XGB)
# =====================================================================

print("\nTraining final XGBoost on full dataset...")

best = get_models()["XGBoost"]
best.fit(X, y, verbose=False)

# =====================================================================
# FEATURE IMPORTANCE
# =====================================================================

imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best.feature_importances_
}).sort_values("Importance", ascending=False)

imp.to_csv(
    os.path.join(OUT_DIR, "feature_importance.csv"),
    index=False
)

top20 = imp.head(20)

plt.figure(figsize=(10, 8))
plt.barh(top20["Feature"], top20["Importance"])
plt.gca().invert_yaxis()
plt.title("Top 20 Feature Importance")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "feature_importance.png"), dpi=200)
plt.close()

# =====================================================================
# STABILITY CHART
# =====================================================================

plt.figure(figsize=(10, 6))

for m in metrics["Model"].unique():
    temp = metrics[metrics["Model"] == m]
    plt.plot(
        temp["Fold"],
        temp["ROC_AUC"],
        marker="o",
        label=m
    )

plt.title("Walk-Forward ROC AUC Stability")
plt.xlabel("Fold")
plt.ylabel("ROC AUC")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "stability_chart.png"), dpi=200)
plt.close()

# =====================================================================
# SECTOR ROBUSTNESS (XGB ONLY)
# =====================================================================

xgb_preds = predictions[predictions["Model"] == "XGBoost"].copy()

sector_stats = xgb_preds.groupby("Sector").apply(
    lambda z: (z["Actual"] == z["Pred"]).mean()
).reset_index()

sector_stats.columns = ["Sector", "Accuracy"]
sector_stats = sector_stats.sort_values("Accuracy", ascending=False)

sector_stats.to_csv(
    os.path.join(OUT_DIR, "sector_robustness.csv"),
    index=False
)

# =====================================================================
# DECISION TREE VISUAL
# =====================================================================

print("Training Decision Tree chart...")

tree = get_models()["DecisionTree"]
tree.fit(X, y)

plt.figure(figsize=(24, 12))

plot_tree(
    tree,
    feature_names=feature_names,
    class_names=["Down", "Up"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "decision_tree.png"), dpi=200)
plt.close()

# =====================================================================
# FINISH
# =====================================================================

print("\n" + "=" * 72)
print("PHASE 2 ULTRA FINAL COMPLETE")
print("=" * 72)
print("Output Folder:", OUT_DIR)

print("""
FILES CREATED:
- model_summary.csv
- walkforward_fold_metrics.csv
- predictions.csv
- feature_importance.csv
- sector_robustness.csv
- feature_importance.png
- stability_chart.png
- decision_tree.png
""")

LOADING DATA
Rows : 1,980,952
Cols : 26

Feature Columns: 21
Target Balance:
target
1    0.507876
0    0.492124
Name: proportion, dtype: float64

WALK-FORWARD ANALYSIS

Running DecisionTree

Running RandomForest

Running XGBoost

MODEL SUMMARY
          Model  Accuracy_mean  Accuracy_std  Precision_mean  Precision_std  \
2       XGBoost       0.514241      0.003507        0.521862       0.020090   
1  RandomForest       0.515554      0.014451        0.517138       0.021571   
0  DecisionTree       0.513419      0.013037        0.516352       0.021433   

   ROC_AUC_mean  ROC_AUC_std  Top10pct_Precision_mean  Top5pct_Precision_mean  \
2      0.520038     0.004782                 0.542814                0.548252   
1      0.518902     0.003586                 0.539921                0.548601   
0      0.510603     0.002833                 0.525307                0.531381   

   RuntimeSec_mean  
2          81.7650  
1         244.4150  
0          45.9475  

Training final XGBoost on ful

In [3]:
# ==============================================================
# BONUS CHART PACK - FINAL UPDATED VERSION
# Uses your actual Phase 2 output folder
# ==============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")
plt.style.use("ggplot")

# ==============================================================
# YOUR FOLDER PATH
# ==============================================================

folder = r"C:\Users\navon\anaconda3\envs\quant\share\jupyter\lab\phase2_ultra_output_20260423_213205"

# ==============================================================
# CHECK FILES
# ==============================================================

required_files = [
    "model_summary.csv",
    "predictions.csv",
    "sector_robustness.csv",
    "walkforward_fold_metrics.csv",
    "feature_importance.csv"
]

print("="*60)
print("CHECKING FILES")
print("="*60)

for f in required_files:
    path = os.path.join(folder, f)
    print(f, "✓" if os.path.exists(path) else "✗")

# ==============================================================
# LOAD DATA
# ==============================================================

model_summary = pd.read_csv(os.path.join(folder, "model_summary.csv"))
pred = pd.read_csv(os.path.join(folder, "predictions.csv"))
sector = pd.read_csv(os.path.join(folder, "sector_robustness.csv"))
folds = pd.read_csv(os.path.join(folder, "walkforward_fold_metrics.csv"))
feat = pd.read_csv(os.path.join(folder, "feature_importance.csv"))

# ==============================================================
# OUTPUT FOLDER
# ==============================================================

bonus = os.path.join(folder, "bonus_charts")
os.makedirs(bonus, exist_ok=True)

print("\nLoaded all files successfully.")
print("Saving charts to:", bonus)

# ==============================================================
# 1. MODEL SCORECARD
# ==============================================================

plt.figure(figsize=(10,6))
plt.bar(model_summary["Model"], model_summary["ROC_AUC_mean"])
plt.title("Model ROC AUC Comparison")
plt.ylabel("ROC AUC")
plt.tight_layout()
plt.savefig(os.path.join(bonus, "01_model_scorecard.png"), dpi=300)
plt.close()

# ==============================================================
# 2. TOP SIGNAL PRECISION
# ==============================================================

plt.figure(figsize=(10,6))

x = np.arange(len(model_summary))
w = 0.35

plt.bar(x-w/2, model_summary["Top10pct_Precision_mean"], width=w, label="Top 10%")
plt.bar(x+w/2, model_summary["Top5pct_Precision_mean"], width=w, label="Top 5%")

plt.xticks(x, model_summary["Model"])
plt.ylabel("Precision")
plt.title("Precision on Strongest Signals")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(bonus, "02_top_signal_precision.png"), dpi=300)
plt.close()

# ==============================================================
# 3. SECTOR BAR CHART
# ==============================================================

sector2 = sector.sort_values("Accuracy", ascending=False).head(15)

plt.figure(figsize=(10,7))
plt.barh(sector2["Sector"], sector2["Accuracy"])
plt.gca().invert_yaxis()
plt.title("Top Sector Robustness")
plt.xlabel("Accuracy")
plt.tight_layout()
plt.savefig(os.path.join(bonus, "03_sector_robustness.png"), dpi=300)
plt.close()

# ==============================================================
# 4. FEATURE PIE
# ==============================================================

top10 = feat.head(10)

plt.figure(figsize=(10,10))
plt.pie(
    top10["Importance"],
    labels=top10["Feature"],
    autopct="%1.1f%%",
    startangle=140
)
plt.title("Top 10 Feature Contribution")
plt.tight_layout()
plt.savefig(os.path.join(bonus, "04_feature_pie.png"), dpi=300)
plt.close()

# ==============================================================
# 5. ROC AUC BY FOLD
# ==============================================================

plt.figure(figsize=(10,6))

for model in folds["Model"].unique():
    temp = folds[folds["Model"] == model]
    plt.plot(temp["Fold"], temp["ROC_AUC"], marker="o", label=model)

plt.title("Fold-by-Fold ROC AUC")
plt.xlabel("Fold")
plt.ylabel("ROC AUC")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(bonus, "05_fold_auc_lines.png"), dpi=300)
plt.close()

# ==============================================================
# 6. PREDICTION PROBABILITY DISTRIBUTION
# ==============================================================

if "Prob" in pred.columns:

    plt.figure(figsize=(10,6))
    plt.hist(pred["Prob"], bins=40)
    plt.title("Prediction Probability Distribution")
    plt.xlabel("Probability of Up Move")
    plt.tight_layout()
    plt.savefig(os.path.join(bonus, "06_probability_distribution.png"), dpi=300)
    plt.close()

# ==============================================================
# 7. MODEL SPEED CHART
# ==============================================================

plt.figure(figsize=(10,6))
plt.bar(model_summary["Model"], model_summary["RuntimeSec_mean"])
plt.title("Average Runtime by Model")
plt.ylabel("Seconds")
plt.tight_layout()
plt.savefig(os.path.join(bonus, "07_model_runtime.png"), dpi=300)
plt.close()

# ==============================================================
# 8. MODEL RANK CSV
# ==============================================================

rank = model_summary.sort_values(
    ["ROC_AUC_mean", "Top5pct_Precision_mean"],
    ascending=False
)

rank.to_csv(os.path.join(bonus, "08_model_rankings.csv"), index=False)

# ==============================================================
# COMPLETE
# ==============================================================

print("\n" + "="*60)
print("BONUS CHART PACK COMPLETE")
print("="*60)

print("""
Files Created:

01_model_scorecard.png
02_top_signal_precision.png
03_sector_robustness.png
04_feature_pie.png
05_fold_auc_lines.png
06_probability_distribution.png
07_model_runtime.png
08_model_rankings.csv
""")

CHECKING FILES
model_summary.csv ✓
predictions.csv ✓
sector_robustness.csv ✓
walkforward_fold_metrics.csv ✓
feature_importance.csv ✓

Loaded all files successfully.
Saving charts to: C:\Users\navon\anaconda3\envs\quant\share\jupyter\lab\phase2_ultra_output_20260423_213205\bonus_charts

BONUS CHART PACK COMPLETE

Files Created:

01_model_scorecard.png
02_top_signal_precision.png
03_sector_robustness.png
04_feature_pie.png
05_fold_auc_lines.png
06_probability_distribution.png
07_model_runtime.png
08_model_rankings.csv

